# Trade Anomaly Detection

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

## Load Data

In [ ]:
data_paths = [
    'data/final_csv/02_trade_anomaly_dl.csv',
    '../data/final_csv/02_trade_anomaly_dl.csv',
    'backend/brain/data/final_csv/02_trade_anomaly_dl.csv',
    'brain/datasets/dump/brain_prev/data_pipeline/data/final_csv/02_trade_anomaly_dl.csv'
]
path = next(p for p in data_paths if os.path.exists(p))
df = pd.read_csv(path)
df.head()

## Exploratory Analysis

In [ ]:
print(f"Total records: {len(df):,}")
if 'anomaly_flag' in df.columns:
    print(df['anomaly_flag'].value_counts(normalize=True))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df['trade_value_usd'].apply(np.log1p).hist(bins=30)
plt.title("Log Trade Value (USD)")

plt.subplot(1, 2, 2)
if 'anomaly_type' in df.columns:
    df['anomaly_type'].value_counts().head(5).plot(kind='barh')
    plt.title("Top Anomaly Types")
plt.tight_layout()
plt.show()

## Anomaly Models (Isolation Forest & XGBoost)

In [ ]:
features = ['trade_value_usd', 'net_weight_kg', 'quantity']
features = [f for f in features if f in df.columns]
if 'unit_price' not in df.columns and 'trade_value_usd' in df.columns and 'quantity' in df.columns:
    df['unit_price'] = df['trade_value_usd'] / (df['quantity'] + 1e-5)
    features.append('unit_price')

X = df[features].fillna(0).values
y = df['anomaly_flag'].values if 'anomaly_flag' in df.columns else np.zeros(len(df))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# isolation forest
iso = IsolationForest(n_estimators=100, contamination=0.08, random_state=42)
iso.fit(X_train)
iso_scores = -iso.decision_function(X_test)

# xgboost
xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
xgb_clf.fit(X_train, y_train)
xgb_scores = xgb_clf.predict_proba(X_test)[:, 1]

## Evaluation

In [ ]:
if len(np.unique(y_test)) > 1:
    print(f"Isolation Forest ROC-AUC: {roc_auc_score(y_test, iso_scores):.4f}")
    print(f"XGBoost PR-AUC: {average_precision_score(y_test, xgb_scores):.4f}")
    print(f"XGBoost ROC-AUC: {roc_auc_score(y_test, xgb_scores):.4f}")

sample_res = pd.DataFrame(X_test[:5], columns=features)
sample_res['anomaly_prob'] = np.round(xgb_scores[:5], 3)
sample_res['actual'] = y_test[:5]
sample_res